# 1장. Tool 기초

**Tool**은 LLM이 외부 시스템(API, DB, 파일 시스템 등)과 상호작용하기 위해 사용하는 함수입니다.  
LLM 모델은 학습 데이터의 시간적 한계(Knowledge Cutoff)가 있어 실시간 날씨, 최신 뉴스 같은 정보를 직접 알지 못합니다.  
Tool을 사용하면 이런 한계를 극복할 수 있습니다.

- `01.ipynb` : Tool 기초 + Tool 응용 1 (사칙연산 Agent)

In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

## Tool 작성법

Tool을 만드는 방법은 4단계입니다.

1. **함수 작성** : 파이썬 함수로 실제 로직 구현
2. **타입 힌팅** : 입력·출력 타입 명시 → LLM이 인자를 정확히 채울 수 있게 됨
3. **docstring** 으로 함수 설명 → LLM이 이 Tool을 *언제* 써야 할지 이해
4. **`@tool` 데코레이터** 추가

> `docstring`은 LLM에게 "이 Tool은 이런 상황에 써라"는 설명서 역할을 합니다. 설명이 정확할수록 LLM이 올바른 Tool을 선택합니다.

In [3]:
# 현재 날씨를 알려주는 tool 함수 예시
@tool
def get_current_weather(location: str) -> str:
  """
  주어진 위치(location)의 현재 날씨 정보를 반환합니다.
  실제 구현에서는 외부 API 호출이 필요합니다.
  """
  # 예시 응답 (실제 구현 시 API 연동 필요)
  return f"{location}의 현재 날씨는 맑음, 22도입니다."

## Tool과 Model 연결 — Agent 생성

`langchain.agents.create_agent`에 모델과 tools 목록을 전달하면 **Agent**가 생성됩니다.  
Agent는 내부적으로 LLM이 어떤 Tool을 언제 호출할지 스스로 결정합니다.

In [4]:
# 에이전트 생성   
agent_with_weather = create_agent(
    model=model,
    tools=[get_current_weather],)


## Agent 실행 및 내부 messages 구조

`invoke()`로 Agent를 실행하면 `messages` 리스트에 다음 순서로 메시지가 쌓입니다.

| 순서 | 메시지 타입 | 내용 |
|:---:|:---|:---|
| 1 | `HumanMessage` | 사용자 질문 |
| 2 | `AIMessage` | Tool 호출 결정 (`content=''`, `tool_calls=[...]`) |
| 3 | `ToolMessage` | Tool 실행 결과 |
| 4 | `AIMessage` | 최종 자연어 답변 |

최종 응답은 `result["messages"][-1].content`로 꺼냅니다.

In [5]:
result = agent_with_weather.invoke({"messages": [
  {"role": "user", "content": "서울 날씨 어때"},
]})

print("Agent의 응답:", result)
print(result['messages'][-1].content)

Agent의 응답: {'messages': [HumanMessage(content='서울 날씨 어때', additional_kwargs={}, response_metadata={}, id='34a0fd3e-2c89-4d8b-9771-a5480f6fda5b'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 66, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DLX0s78bIF4pRnafczJ6a4oO62pUJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d0c14-7741-7aa3-8367-65b275da928f-0', tool_calls=[{'name': 'get_current_weather', 'args': {'location': '서울'}, 'id': 'call_ao1PVW3cUkvmYrlXB5hcCQGP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 66, 'output_to

## Tool 응용 1 — 여러 Tool + System Message

여러 개의 Tool을 동시에 장착하고, **System Message**로 LLM의 행동 방침을 지정합니다.

LLM은 확률 기반 추론을 하기 때문에, 간단한 계산 문제는 Tool을 쓰지 않고 직접 답하려 할 수 있습니다.  
이를 방지하려면 `system_prompt`에 **"반드시 Tool을 사용하라"** 는 지시를 명시해야 합니다.

In [6]:
# 사칙연산 tool 함수 정의
@tool
def add(a: int, b: int) -> float:
  """두 수의 합을 반환합니다."""
  return a + b

@tool
def divide(a: float, b: float) -> float:
  """a를 b로 나눈 값을 반환합니다. (b가 0이면 예외 발생)"""
  if b == 0:
    raise ValueError("0으로 나눌 수 없습니다.")
  return a / b

@tool
def multiply(a: float, b: float) -> float:
  """두 수의 곱을 반환합니다."""
  return a * b


In [7]:
# AI는 확률 기반으로 추론하는데, tool를 사용하지 않고도 답을 할 수 있다고 판단하게 되면 tools를 사용하지 않을 수 있다
# 이런 경우를 방지하려면, system 메시지에 반드시 tool를 사용하도록 지시하는 내용을 추가해야 합니다.
agent_with_calculator = create_agent(
    model=model,
    tools=[add, divide, multiply],
    system_prompt="당신은 계산기입니다. 반드시 도구를 사용하여 계산을 수행해야 합니다. 절대 도구를 사용하지 않고 대답하지 마세요.")


result = agent_with_calculator.invoke({"messages": [
  {"role": "user", "content": "32 + 5 * 100를 계산해줘."},
]})

print("Agent의 응답:", result) 

Agent의 응답: {'messages': [HumanMessage(content='32 + 5 * 100를 계산해줘.', additional_kwargs={}, response_metadata={}, id='aebb5152-4e73-4a3a-aae7-755e5cc5fa19'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 156, 'total_tokens': 206, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DLX14gzZaqGPAEtBv9QYs5aUs9WQd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d0c14-aba3-71d0-85e6-f30502f2442b-0', tool_calls=[{'name': 'multiply', 'args': {'a': 5, 'b': 100}, 'id': 'call_1yNZNOFpbHQ4BQQ6jjyaVrTK', 'type': 'tool_call'}, {'name': 'add', 'args': {'a': 32, 'b': 0}, 'id': 'call_yubeMODRNvG2T

## Agent 다단계 추론 결과 확인

`"32 + 5 * 100"` 연산 시 Agent가 수학적 우선순위에 따라 단계별로 Tool을 호출하는 것을 확인할 수 있습니다.

1. `multiply(5, 100)` → 500
2. `add(32, 500)` → 532

`result["messages"]`를 출력하면 각 단계의 `AIMessage(tool_calls=...)` → `ToolMessage(content=...)` 흐름을 볼 수 있습니다.